In [7]:
if 'spark' in globals():
    spark.stop()

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.sql.legacy.parquet.nanosAsLong", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/13 19:38:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
Pu = "s3a://end-to-end-streaming-data-platform-bronze/postgres/ingestion_date=2026-07-13-Jul/users.parquet"

In [3]:
dfu = spark.read.parquet(Pu).cache()
dfu.createOrReplaceTempView("users_table")

26/07/13 19:42:16 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [6]:
dfu.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- last_login: timestamp (nullable = true)



In [7]:
dfu.show(5)

+--------------------+--------------------+-------+--------------+--------------------+--------------------+
|             user_id|               email|country|account_status|          created_at|          last_login|
+--------------------+--------------------+-------+--------------+--------------------+--------------------+
|a760dcac-c679-49a...|ychristensen@exam...|    EGY|       deleted|2025-09-26 08:07:...|                NULL|
|b48b3038-39d2-417...|msullivan@example...|    EGY|        active|2025-12-02 15:36:...|2025-12-02 15:36:...|
|a6697d49-92fb-4fb...| ytaylor@example.net|    KSA|     suspended|2025-01-01 05:56:...|                NULL|
|effa0d9c-8b2d-402...|gatesjessica@exam...|    KSA|        active|2026-01-26 15:38:...|2026-02-05 15:38:...|
|d28a3a9e-e34f-4e7...|rosetimothy@examp...|    EGY|        active|2024-11-17 09:23:...|2024-12-11 09:23:...|
+--------------------+--------------------+-------+--------------+--------------------+--------------------+
only showing top 5 

In [8]:
user_silver_transformation= """
WITH users_clean AS (
    SELECT
        user_id,
        email,
        COALESCE(
            MAP('KSA', 'Saudi Arabia',
                'ECY', 'Egypt',
                'UAE', 'United Arab Emirates',
                'QAT', 'Qatar',
                'KWT', 'Kuwait',
                'jOR', 'Jordan',
                'MAR', 'Morocco'
            )[country],
            'Unknown'
        ) AS country_name,
        account_status,
        created_at AS created_at_timestamp,
        last_login AS last_login_timestamp
    FROM users_table
)
SELECT *,
    CASE WHEN last_login_timestamp < created_at_timestamp THEN NULL ELSE last_login_timestamp END AS last_login_validated,
    CASE WHEN last_login_timestamp < created_at_timestamp THEN 1 ELSE 0 END AS is_login_before_creation,
    CASE WHEN email IS NULL AND last_login_timestamp IS NOT NULL THEN "needs_review" ELSE NULL
    END AS review_status,
    CASE WHEN email IS NOT NULL THEN 1 ELSE 0 END AS has_email,
    CASE WHEN last_login_timestamp IS NOT NULL THEN 1 ELSE 0 END AS has_logged_in,
    CAST(created_at_timestamp AS DATE) AS created_date,
    CAST(last_login_timestamp AS DATE) AS last_login_date
FROM users_clean
"""

In [9]:
users_df = spark.sql(user_silver_transformation)

In [ ]:
## Row Count Check

In [11]:
assert users_df.count() == dfu.count()

In [12]:
assert subscriptions_df.count() == dfs.count()

In [13]:
assert payments_df.count() == dfp.count()

In [ ]:
# ID Match Check 
# Using collect() here is safe only 1 column and 1000 rows

In [17]:
raw_ids = set(row["subscription_id"] for row in dfs.select("subscription_id").collect())
silver_ids = set(row["subscription_id"] for row in subscriptions_df.select("subscription_id").collect())
assert raw_ids == silver_ids

In [ ]:
raw_ids = set(row["user_id"] for row in dfu.select("user_id").collect())
silver_ids = set(row["user_id"] for row in users_df.select("user_id").collect())
assert raw_ids == silver_ids

In [20]:
raw_ids = set(row["payment_id"] for row in dfp.select("payment_id").collect())
silver_ids = set(row["payment_id"] for row in payments_df.select("payment_id").collect())
assert raw_ids == silver_ids

In [ ]:
# Duplication Check

In [22]:
assert users_df.groupBy("user_id").count().filter("count > 1").count() == 0

In [23]:
assert subscriptions_df.groupBy("subscription_id").count().filter("count > 1").count() == 0

In [24]:
assert payments_df.groupBy("payment_id").count().filter("count > 1").count() == 0

In [ ]:
## Business Logic Validation

In [25]:
# has_email
assert users_df.filter(
    ((F.col("email").isNotNull()) & (F.col("has_email") != 1)) |
    ((F.col("email").isNull()) & (F.col("has_email") != 0)) 
).count() == 0
# has_logged_in
assert users_df.filter(
    ((F.col("last_login_timestamp").isNotNull()) & (F.col("has_logged_in") != 1)) |
    ((F.col("last_login_timestamp").isNull()) & (F.col("has_logged_in") != 0))
).count() == 0
# is_login_before_creation
assert users_df.filter(
    (F.col("last_login_timestamp") < F.col("created_at_timestamp")) & (F.col("is_login_before_creation") != 1)
).count() == 0
# last_login_validated
assert users_df.filter(
    (F.col("is_login_before_creation") == 1)  & (F.col("last_login_validated").isNotNull())
).count() == 0

In [27]:
# is_end_date_before_start_date
assert subscriptions_df.filter(
    (F.col("end_date_timestamp") < F.col("start_date_timestamp")) & (F.col("is_end_date_before_start_date") != 1)
).count() == 0

assert subscriptions_df.filter(
    ((F.col("end_date_timestamp") >= F.col("start_date_timestamp")) | (F.col("end_date_timestamp").isNull())) &
    (F.col("is_end_date_before_start_date") != 0)
).count() == 0

# end_date_validated
assert subscriptions_df.filter(
    (F.col("is_end_date_before_start_date") == 1) & (F.col("end_date_validated").isNotNull())
).count() == 0

# review_status
assert subscriptions_df.filter(
    (F.col("is_end_date_before_start_date") == 1) & (F.col("review_status") != "needs_review")
).count() == 0

In [29]:
# Join silver with bronze on payment_id to compare original vs transformed amount

joined = payments_df.alias("s").join(
    dfp.alias("b"), on="payment_id"
)

# success -> ABS(original amount)
assert joined.filter(
    (F.col("b.payment_status") == "success") & (F.col("s.amount") != F.abs(F.col("b.amount")))
).count() == 0

# failed -> 0
assert joined.filter(
    (F.col("b.payment_status") == "failed") & (F.col("s.amount") != 0)
).count() == 0

# refunded -> -ABS(original amount)
assert joined.filter(
    (F.col("b.payment_status") == "refunded") & (F.col("s.amount") != -F.abs(F.col("b.amount")))
).count() == 0

In [ ]:
# Unexpected Nulls Check 

In [30]:
assert users_df.filter(F.col("country_name").isNull()).count() == 0
assert users_df.filter(F.col("is_login_before_creation").isNull()).count() == 0
assert users_df.filter(F.col("has_email").isNull()).count() == 0
assert users_df.filter(F.col("has_logged_in").isNull()).count() == 0

In [31]:
assert subscriptions_df.filter(F.col("is_end_date_before_start_date").isNull()).count() == 0

In [32]:
# amount should never be null if payment_status is one of the known values
assert payments_df.filter(
    (F.col("payment_status").isin("success", "failed", "refunded")) &
    (F.col("amount").isNull())
).count() == 0

# Bonus: catch any unexpected payment_status values entirely
assert payments_df.filter(
    ~F.col("payment_status").isin("success", "failed", "refunded")
).count() == 0

In [ ]:
## Null after Casting

In [33]:
expected_schema_users = {
    "user_id": "string",
    "email": "string",
    "country_name": "string",
    "account_status": "string",
    "created_at_timestamp": "timestamp",
    "last_login_timestamp": "timestamp",
    "last_login_validated": "timestamp",
    "is_login_before_creation": "int",
    "review_status": "string",
    "has_email": "int",
    "has_logged_in": "int",
    "created_date": "date",
    "last_login_date": "date",
}

actual_schema_users = {f.name: f.dataType.simpleString() for f in users_df.schema.fields}

assert actual_schema_users == expected_schema_users

In [34]:
expected_schema_subscriptions = {
    "subscription_id": "string",
    "user_id": "string",
    "plan_type": "string",
    "start_date_timestamp": "timestamp",
    "end_date_timestamp": "timestamp",
    "is_active": "boolean",
    "auto_renew": "boolean",
    "end_date_validated": "timestamp",
    "is_end_date_before_start_date": "int",
    "review_status": "string",
    "start_date": "date",
    "end_date": "date",
}

actual_schema_subscriptions = {f.name: f.dataType.simpleString() for f in subscriptions_df.schema.fields}

assert actual_schema_subscriptions == expected_schema_subscriptions

In [35]:
expected_schema_payments = {
    "payment_id": "string",
    "subscription_id": "string",
    "amount": "double",
    "currency": "string",
    "payment_date_timestamp": "timestamp",
    "payment_date": "date",
    "payment_status": "string",
}

actual_schema_payments = {f.name: f.dataType.simpleString() for f in payments_df.schema.fields}

assert actual_schema_payments == expected_schema_payments

In [ ]:
## Edge Case: exact count of "end_date before start_date" must match known EDA finding

In [36]:
assert subscriptions_df.filter(F.col("is_end_date_before_start_date") == 1).count() == 29

In [37]:
# All "success" amounts must be positive after transformation
assert payments_df.filter(
    (F.col("payment_status") == "success") & (F.col("amount") < 0)
).count() == 0

# All "failed" amounts must be exactly 0
assert payments_df.filter(
    (F.col("payment_status") == "failed") & (F.col("amount") != 0)
).count() == 0

# All "refunded" amounts must be negative or zero
assert payments_df.filter(
    (F.col("payment_status") == "refunded") & (F.col("amount") > 0)
).count() == 0

In [38]:
# Case 1: last_login before created_at → must equal known EDA count (27)
assert users_df.filter(F.col("is_login_before_creation") == 1).count() == 27

# Case 2: email is null AND last_login is not null → must equal known EDA count (19)
assert users_df.filter(
    (F.col("email").isNull()) & (F.col("last_login_timestamp").isNotNull())
).count() == 19

# Bonus: confirm these 19 cases are correctly flagged as needs_review
assert users_df.filter(
    (F.col("email").isNull()) & (F.col("last_login_timestamp").isNotNull()) &
    (F.col("review_status") != "needs_review")
).count() == 0

In [ ]:
# OUTPUT

In [7]:
users_df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country_name: string (nullable = false)
 |-- account_status: string (nullable = true)
 |-- created_at_timestamp: timestamp (nullable = true)
 |-- last_login_timestamp: timestamp (nullable = true)
 |-- last_login_validated: timestamp (nullable = true)
 |-- is_login_before_creation: integer (nullable = false)
 |-- review_status: string (nullable = true)
 |-- has_email: integer (nullable = false)
 |-- has_logged_in: integer (nullable = false)
 |-- created_date: date (nullable = true)
 |-- last_login_date: date (nullable = true)



In [10]:
users_df.write \
    .mode("overwrite") \
    .parquet("s3a://end-to-end-streaming-data-platform-silver/postgres/users_silver.parquet")

In [15]:
Ps = "s3a://end-to-end-streaming-data-platform-bronze/postgres/ingestion_date=2026-07-13-Jul/subscription.parquet"
Pp = "s3a://end-to-end-streaming-data-platform-bronze/postgres/ingestion_date=2026-07-13-Jul/payments.parquet"

In [16]:
dfs = spark.read.parquet(Ps).cache()
dfp = spark.read.parquet(Pp).cache()
dfs.createOrReplaceTempView("subscriptions_table")
dfp.createOrReplaceTempView("payments_table")

In [17]:
subscription_silver_transformation= """
WITH sub_clean AS (    
    SELECT
    subscription_id,
    user_id,
    plan_type,
    start_date AS start_date_timestamp,
    end_date AS end_date_timestamp,
    is_active,
    auto_renew
    FROM subscriptions_table
)
SELECT *,
    CASE WHEN end_date_timestamp < start_date_timestamp THEN NULL
    ELSE end_date_timestamp END AS end_date_validated,
    CASE WHEN end_date_timestamp < start_date_timestamp THEN 1 
    ELSE 0 END AS is_end_date_before_start_date,
    CASE WHEN end_date_timestamp IS NOT NULL AND end_date_timestamp < start_date_timestamp THEN "needs_review"
    ELSE NULL END AS review_status,
    CAST(start_date_timestamp AS DATE) AS start_date,
    CAST(end_date_timestamp AS DATE) AS end_date
FROM sub_clean
"""

In [18]:
subscriptions_df = spark.sql(subscription_silver_transformation)

In [19]:
users_df.write \
    .mode("overwrite") \
    .parquet("s3a://end-to-end-streaming-data-platform-silver/postgres/subscriptions_silver.parquet")

In [20]:
payments_silver_transformation= """
SELECT
    payment_id,
    subscription_id,
    CASE
        WHEN payment_status = "success" THEN ABS(amount)
        WHEN payment_status = "failed" THEN 0
        WHEN payment_status = "refunded" THEN -ABS(amount)
    END AS amount,
    currency,
    payment_date AS payment_date_timestamp,
    CAST(payment_date AS DATE) AS payment_date,
    payment_status
FROM payments_table
"""

In [21]:
payments_df = spark.sql(payments_silver_transformation)

In [22]:
users_df.write \
    .mode("overwrite") \
    .parquet("s3a://end-to-end-streaming-data-platform-silver/postgres/payments_silver.parquet")